<a href="https://colab.research.google.com/github/magnagomatheus/SpaceshipTitanicProblem/blob/main/SpaceshipTitanicProblem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise dados
# Definir colunas importantes
# Processamento e Escalonamento
# Validação cruzada
# Pipeline
# Optuna? talvez depois muito pesado
# executar teste vulgo .predict()

IMPORT DATABASE

In [ ]:
import pandas as pd

df = pd.read_csv('train.csv')
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Null values verification

In [ ]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


,0
PassengerId,0
HomePlanet,201
CryoSleep,217
Cabin,199
Destination,182
Age,179
VIP,203
RoomService,181
FoodCourt,183
ShoppingMall,208


Getting the target column and removing it from the train part of dataframe

In [ ]:
y = df['Transported']
X = df.drop('Transported', axis=1)

In [ ]:
len(y)

8693

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
dtypes: float64(6), object(7)
memory usage: 883.0+ KB


Doing some automatization to garantee no rewriting code and that the treatments be done

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
# Removing undesired columns and leaving only the desired attributes
class DesiredAttributes(BaseEstimator, TransformerMixin):
  def __init__(self):
    self.undesiredColumns = ['PassengerId', 'Name', 'Cabin']
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X.drop(columns=self.undesiredColumns, axis=1)

desired_attributes = DesiredAttributes()
Xdrop = desired_attributes.fit_transform(X)
Xdrop.tail()


,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
8688,Europa,False,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0
8689,Earth,True,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0
8690,Earth,False,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0
8691,Europa,False,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0
8692,Europa,False,TRAPPIST-1e,44.0,False,126.0,4688.0,0.0,0.0,12.0


Automatization of Splitting numeric elements and treating then

In [ ]:
class NumericAttributes(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X.select_dtypes('number')

numeric_attributes = NumericAttributes()
Xnum = numeric_attributes.fit_transform(Xdrop)
Xnum.tail()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
8688,41.0,0.0,6819.0,0.0,1643.0,74.0
8689,18.0,0.0,0.0,0.0,0.0,0.0
8690,26.0,0.0,0.0,1872.0,1.0,0.0
8691,32.0,0.0,1049.0,0.0,353.0,3235.0
8692,44.0,126.0,4688.0,0.0,0.0,12.0


Creating pipeline to Treat null values and Scale the values from numeric part

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipenum = Pipeline([
    ('numericAttributes', NumericAttributes()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
Xnum = pipenum.fit_transform(Xdrop)
Xnum[-5:]

array([[ 0.85140975, -0.33310466,  3.99233603, -0.28357863,  1.18917257,
        -0.19775108],
       [-0.75243062, -0.33310466, -0.28102673, -0.28357863, -0.2706259 ,
        -0.26300329],
       [-0.1945731 , -0.33310466, -0.28102673,  2.84699872, -0.26973741,
        -0.26300329],
       [ 0.22382004, -0.33310466,  0.37636549, -0.28357863,  0.04301309,
         2.58957649],
       [ 1.06060632, -0.14233462,  2.656871  , -0.28357863, -0.2706259 ,
        -0.25242185]])

Automatization of Splitting categoric attributes and treating then



In [ ]:
class CategoricAttributes(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X.select_dtypes('object')

cat_attributes = CategoricAttributes()
Xcat = cat_attributes.fit_transform(Xdrop)
Xcat.tail()

,HomePlanet,CryoSleep,Destination,VIP
8688,Europa,False,55 Cancri e,True
8689,Earth,True,PSO J318.5-22,False
8690,Earth,False,TRAPPIST-1e,False
8691,Europa,False,55 Cancri e,False
8692,Europa,False,TRAPPIST-1e,False


Creating pipeline that get the categoric attributes, treat your null values and apply one hot encoding that transform each categoric in a column with 0 or 1 (it is or not)

In [ ]:
pipecat = Pipeline([
    ('catAttributes', CategoricAttributes()),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # handle_unkown to fix the error, but it is better to get the column cabin
    # and extract it's classifications to a unique exclusive column for each classification.
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
XcatTreated = pipecat.fit_transform(Xdrop)
XcatTreated.toarray()[-5:]

array([[0., 1., 0., 1., 0., 1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1., 0., 1., 0., 1., 0.],
       [1., 0., 0., 1., 0., 0., 0., 1., 1., 0.],
       [0., 1., 0., 1., 0., 1., 0., 0., 1., 0.],
       [0., 1., 0., 1., 0., 0., 0., 1., 1., 0.]])

Pipeline to get the numeric and categoric parts together.

In [ ]:
from sklearn.pipeline import FeatureUnion

assemblecharacteristics = FeatureUnion([
    ('pipenum', pipenum),
    ('pipecat', pipecat)
])
Xtreated = assemblecharacteristics.fit_transform(Xdrop)
Xtreated[-5:]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 50 stored elements and shape (5, 16)>

Using DenseTransform to fix a conflict with RandomForestAlgorithm data format (sparse matrix to standard numpy matrix)

In [ ]:
class DenseTransformer(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X.toarray()

Pipeline for preprocessing data format with DenseTransform

In [ ]:
preprocessing = Pipeline([
    ('desiredAttributes', DesiredAttributes()),
    ('assemblecharacteristics', assemblecharacteristics),
    ('denseTransform', DenseTransformer())
])
Xtreated = preprocessing.fit_transform(X)
Xtreated[-5:]

array([[ 0.85140975, -0.33310466,  3.99233603, -0.28357863,  1.18917257,
        -0.19775108,  0.        ,  1.        ,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [-0.75243062, -0.33310466, -0.28102673, -0.28357863, -0.2706259 ,
        -0.26300329,  1.        ,  0.        ,  0.        ,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  1.        ,
         0.        ],
       [-0.1945731 , -0.33310466, -0.28102673,  2.84699872, -0.26973741,
        -0.26300329,  1.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  1.        ,  1.        ,
         0.        ],
       [ 0.22382004, -0.33310466,  0.37636549, -0.28357863,  0.04301309,
         2.58957649,  0.        ,  1.        ,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ,
         0.        ],
       [ 1.06060632, -0.14233462,  2

Pipeline for the classifier algorithm Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = Pipeline([
    ('preprocessing', preprocessing),
    ('classifier', RandomForestClassifier())
])
model.fit(X, y)

Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('desiredAttributes', DesiredAttributes()),
                                 ('assemblecharacteristics',
                                  FeatureUnion(transformer_list=[('pipenum',
                                                                  Pipeline(steps=[('numericAttributes',
                                                                                   NumericAttributes()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='median')),
                                                                                  ('scaler',
                                                                                   StandardScaler())])),
                                                                 ('pipecat',
                                                                  Pipeline(steps=[('catAttributes',
                                                                                   CategoricAttributes()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='most_frequent')),
                                                                                  ('encoder',
                                                                                   OneHotEncoder(handle_unknown='ignore'))]))])),
                                 ('denseTransform', DenseTransformer())])),
                ('classifier', RandomForestClassifier())])

Getting the results and scores

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X)
accuracy_score(y, y_pred)

/usr/local/lib/python3.13/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


0.9292534222937996

Applying cross validation to see the real model generalization performance

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
scores.mean
print(scores)

/usr/local/lib/python3.13/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instanc

[0.78435883 0.78435883 0.78148361 0.79344074 0.79228999]


/usr/local/lib/python3.13/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Final test

In [ ]:
Xtest = pd.read_csv('test.csv')
y_pred = model.predict(Xtest)
y_pred

/usr/local/lib/python3.13/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


array([ True, False,  True, ...,  True,  True, False])

Submission of the results

In [ ]:
# Criar o DataFrame de submissão estruturado com os IDs originais do arquivo de teste
submission = pd.DataFrame({
    'PassengerId': Xtest['PassengerId'],
    'Transported': y_pred
})

# Salvar em formato CSV pronto para o Kaggle
submission.to_csv('submission.csv', index=False)

# Exibir as primeiras linhas e confirmar o formato
display(submission.head())

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False
